# HW1- Crawling and Scrapping Data from books related to Science for pre-training a Science-GPT

In [ ]:
import os
import requests
from bs4 import BeautifulSoup
import re
from tqdm import tqdm
import hashlib
import time

In [ ]:
# Configuration
BASE_URL = "https://www.gutenberg.org"
BOOKSHELVES = {
    "physics": "https://www.gutenberg.org/ebooks/bookshelf/667",
    "mathematics": "https://www.gutenberg.org/ebooks/bookshelf/672",
    "chemistry": "https://www.gutenberg.org/ebooks/bookshelf/668",
    "biology": "https://www.gutenberg.org/ebooks/bookshelf/669",
    "geology": "https://www.gutenberg.org/ebooks/bookshelf/670",
    "technology": "https://www.gutenberg.org/ebooks/bookshelf/671",
    "Environment": "https://www.gutenberg.org/ebooks/bookshelf/685"
}
OUTPUT_DIR = "gutenberg_science"
BATCH_SIZE_MB = 5  # Size of each batch file in MB
MAX_RETRIES = 3
DELAY_BETWEEN_REQUESTS = 1  # seconds

In [ ]:
os.makedirs(OUTPUT_DIR, exist_ok=True)

def clean_gutenberg_text(text):
    """Clean Gutenberg text to extract only the book content."""
    # Remove Gutenberg header/footer
    start_patterns = [
        r"\*\*\* START OF.*?\*\*\*",
        r"START OF THIS PROJECT GUTENBERG.*?END OF.*?INFORMATION",
        r"PROJECT GUTENBERG.*?START OF THE.*?\*\*\*"
    ]
    end_patterns = [
        r"\*\*\* END OF.*?\*\*\*",
        r"End of the Project Gutenberg.*",
        r"END OF THIS PROJECT GUTENBERG.*"
    ]

    # Apply start patterns
    for pattern in start_patterns:
        match = re.search(pattern, text, re.IGNORECASE | re.DOTALL)
        if match:
            text = text[match.end():]
            break

    # Apply end patterns
    for pattern in end_patterns:
        match = re.search(pattern, text, re.IGNORECASE | re.DOTALL)
        if match:
            text = text[:match.start()]
            break

    # Clean up formatting
    text = re.sub(r"\r\n", "\n", text)  # Normalize line endings
    text = re.sub(r"\n{3,}", "\n\n", text)  # Reduce excessive whitespace
    text = re.sub(r"[ \t]+", " ", text)  # Normalize spaces
    text = re.sub(r"^\s+|\s+$", "", text, flags=re.MULTILINE)  # Strip line whitespace

    # Remove common Gutenberg artifacts
    text = re.sub(r"^Transcriber's Note:.*?$", "", text, flags=re.MULTILINE | re.IGNORECASE)
    text = re.sub(r"\[Illustration.*?\]", "", text, flags=re.IGNORECASE)
    text = re.sub(r"\[pg \d+\]", "", text, flags=re.IGNORECASE)

    return text.strip()

def get_text_hash(text):
    """Generate hash for duplicate detection."""
    return hashlib.md5(text.encode('utf-8')).hexdigest()

def get_books_from_page(url):
    """Return list of book links from a bookshelf page."""
    try:
        time.sleep(DELAY_BETWEEN_REQUESTS)
        r = requests.get(url, timeout=10)
        r.raise_for_status()
        soup = BeautifulSoup(r.text, "html.parser")

        books = []
        for a in soup.select("li.booklink a.link"):
            href = a.get('href', '')
            if href.startswith("/ebooks/"):
                books.append(BASE_URL + href)

        return books
    except Exception as e:
        print(f"Error fetching page {url}: {e}")
        return []

def download_text_file(text_url):
    """Download and return text content from URL."""
    for attempt in range(MAX_RETRIES):
        try:
            time.sleep(DELAY_BETWEEN_REQUESTS)
            response = requests.get(text_url, timeout=30)
            response.raise_for_status()
            return response.text
        except Exception as e:
            if attempt == MAX_RETRIES - 1:
                raise e
            time.sleep(2 ** attempt)

def scrape_book(book_url, seen_hashes):
    """Download one book and return cleaned text if unique."""
    book_id = book_url.split("/")[-1]

    try:
        time.sleep(DELAY_BETWEEN_REQUESTS)
        r = requests.get(book_url, timeout=10)
        r.raise_for_status()
        soup = BeautifulSoup(r.text, "html.parser")

        # Find the best text file link
        text_candidates = []
        for a in soup.select("a"):
            href = a.get("href", "")
            if any(ext in href.lower() for ext in [".txt", ".txt.utf-8"]):
                if href.startswith("/"):
                    href = BASE_URL + href
                text_candidates.append(href)

        # Try to download text
        book_text = None
        if text_candidates:
            # Prefer UTF-8 encoded files
            utf8_files = [url for url in text_candidates if "utf-8" in url.lower()]
            preferred_urls = utf8_files if utf8_files else text_candidates

            for text_url in preferred_urls:
                try:
                    book_text = download_text_file(text_url)
                    break
                except:
                    continue

        if not book_text:
            print(f"No text version found for {book_id}")
            return None

        # Clean the text
        cleaned_text = clean_gutenberg_text(book_text)

        # Skip if text is too short
        if len(cleaned_text) < 1000:
            print(f"Text too short for {book_id}, skipping")
            return None

        # Check for duplicates
        text_hash = get_text_hash(cleaned_text)
        if text_hash in seen_hashes:
            print(f"Duplicate content detected for {book_id}, skipping")
            return None

        seen_hashes.add(text_hash)
        return {
            'id': book_id,
            'text': cleaned_text,
            'hash': text_hash
        }

    except Exception as e:
        print(f"Failed to download {book_url}: {e}")
        return None

def create_batch_files(books_data):
    """Combine books into batch files of specified size."""
    current_batch = ""
    batch_num = 1
    current_size_mb = 0

    print(f"\nCreating batch files (target size: {BATCH_SIZE_MB}MB each)")

    for book_data in tqdm(books_data, desc="Creating batches"):
        book_text = f"\n\n=== BOOK ID: {book_data['id']} ===\n\n{book_data['text']}\n\n"
        book_size_mb = len(book_text.encode('utf-8')) / (1024 * 1024)

        # If adding this book would exceed batch size, save current batch
        if current_size_mb + book_size_mb > BATCH_SIZE_MB and current_batch:
            batch_filename = os.path.join(OUTPUT_DIR, f"batch_{batch_num:03d}.txt")
            with open(batch_filename, "w", encoding="utf-8") as f:
                f.write(current_batch.strip())
            print(f"Saved {batch_filename} ({current_size_mb:.1f}MB)")

            current_batch = ""
            current_size_mb = 0
            batch_num += 1

        current_batch += book_text
        current_size_mb += book_size_mb

    # Save the last batch
    if current_batch:
        batch_filename = os.path.join(OUTPUT_DIR, f"batch_{batch_num:03d}.txt")
        with open(batch_filename, "w", encoding="utf-8") as f:
            f.write(current_batch.strip())
        print(f"Saved {batch_filename} ({current_size_mb:.1f}MB)")

    return batch_num

In [ ]:
def scrape_bookshelf(name, base_url):
    """Scrape all books from a bookshelf."""
    print(f"\nScraping {name} bookshelf...")

    all_books = []
    start_index = 0
    page = 1

    while True:
        url = base_url if start_index == 0 else f"{base_url}?start_index={start_index}"
        books = get_books_from_page(url)

        if not books:
            break

        print(f"{name} page {page} (start_index={start_index}): {len(books)} books")
        all_books.extend(books)
        start_index += 25
        page += 1

    print(f"Total {name} books discovered: {len(all_books)}")
    return all_books

def main():
    """Main scraping function."""
    print("Starting Gutenberg Science Library Scraper")
    print("Categories: Physics, Mathematics, Chemistry, Biology, Geology, Technology, Environmental")
    print(f"Output directory: {OUTPUT_DIR}")

    # Collect all book URLs
    all_book_urls = []
    for name, url in BOOKSHELVES.items():
        books = scrape_bookshelf(name, url)
        all_book_urls.extend(books)

    # Remove duplicate URLs
    unique_urls = list(set(all_book_urls))
    print(f"\nTotal unique book URLs: {len(unique_urls)}")

    # Download and process books
    seen_hashes = set()
    books_data = []

    print("\nDownloading and processing books...")
    for book_url in tqdm(unique_urls, desc="Processing books"):
        book_data = scrape_book(book_url, seen_hashes)
        if book_data:
            books_data.append(book_data)

    print(f"\nSuccessfully processed {len(books_data)} unique books")

    # Create batch files
    if books_data:
        num_batches = create_batch_files(books_data)

        # Calculate and print comprehensive size statistics
        total_size_mb = sum(len(book['text'].encode('utf-8')) for book in books_data) / (1024 * 1024)

        print(f"\nComplete! Created {num_batches} batch files in {OUTPUT_DIR}/")
        print(f"\nFINAL STATISTICS:")
        print(f"Combined text size: {total_size_mb:.2f} MB")
        print(f"Total books processed: {len(books_data)}")
        print(f"Average book size: {total_size_mb/len(books_data):.2f} MB")
        print(f"Number of batch files: {num_batches}")
        print(f"Average batch size: {total_size_mb/num_batches:.2f} MB")

        # Also print in GB if over 1000 MB
        if total_size_mb > 1000:
            print(f"Combined text size: {total_size_mb/1024:.2f} GB")
    else:
        print("No books were successfully processed")

if __name__ == "__main__":
    main()

Starting Gutenberg Science Library Scraper
Categories: Physics, Mathematics, Chemistry, Biology, Geology, Technology, Environmental
Output directory: gutenberg_science

Scraping physics bookshelf...
physics page 1 (start_index=0): 25 books
physics page 2 (start_index=25): 25 books
physics page 3 (start_index=50): 25 books
physics page 4 (start_index=75): 25 books
physics page 5 (start_index=100): 25 books
physics page 6 (start_index=125): 25 books
physics page 7 (start_index=150): 25 books
physics page 8 (start_index=175): 25 books
physics page 9 (start_index=200): 25 books
physics page 10 (start_index=225): 25 books
physics page 11 (start_index=250): 25 books
physics page 12 (start_index=275): 25 books
physics page 13 (start_index=300): 25 books
physics page 14 (start_index=325): 25 books
physics page 15 (start_index=350): 25 books
physics page 16 (start_index=375): 25 books
physics page 17 (start_index=400): 25 books
physics page 18 (start_index=425): 25 books
physics page 19 (start_

Processing books:  44%|████▍     | 1740/3976 [1:30:15<1:51:13,  2.98s/it]

Text too short for 672, skipping


Processing books:  53%|█████▎    | 2116/3976 [1:49:30<1:17:09,  2.49s/it]

No text version found for 7825


...

Saved gutenberg_science/batch_274.txt (4.4MB)
Saved gutenberg_science/batch_275.txt (4.9MB)
Saved gutenberg_science/batch_276.txt (3.7MB)
Saved gutenberg_science/batch_277.txt (4.7MB)
Saved gutenberg_science/batch_278.txt (4.6MB)
Saved gutenberg_science/batch_279.txt (4.9MB)
Saved gutenberg_science/batch_280.txt (4.5MB)
Saved gutenberg_science/batch_281.txt (4.9MB)
Saved gutenberg_science/batch_282.txt (4.7MB)
Saved gutenberg_science/batch_283.txt (4.7MB)


Creating batches:  98%|█████████▊| 3875/3971 [00:07<00:00, 608.56it/s]

Saved gutenberg_science/batch_284.txt (4.5MB)
Saved gutenberg_science/batch_285.txt (4.5MB)
Saved gutenberg_science/batch_286.txt (4.8MB)
Saved gutenberg_science/batch_287.txt (4.8MB)
Saved gutenberg_science/batch_288.txt (4.3MB)
Saved gutenberg_science/batch_289.txt (4.8MB)
Saved gutenberg_science/batch_290.txt (5.0MB)
Saved gutenberg_science/batch_291.txt (3.5MB)
Saved gutenberg_science/batch_292.txt (4.8MB)
Saved gutenberg_science/batch_293.txt (4.8MB)
Saved gutenberg_science/batch_294.txt (4.8MB)


Creating batches: 100%|██████████| 3971/3971 [00:07<00:00, 517.74it/s]


Saved gutenberg_science/batch_295.txt (4.9MB)
Saved gutenberg_science/batch_296.txt (4.8MB)
Saved gutenberg_science/batch_297.txt (1.5MB)

Complete! Created 297 batch files in gutenberg_science/

FINAL STATISTICS:
Combined text size: 1369.99 MB
Total books processed: 3971
Average book size: 0.34 MB
Number of batch files: 297
Average batch size: 4.61 MB
Combined text size: 1.34 GB


# Estimating total size and related statistics

In [ ]:
import os
import glob

def calculate_total_data_length(data_directory="gutenberg_science"):
    """Calculate total length of all text data in characters and other metrics."""

    # Find all .txt files in the directory
    txt_files = glob.glob(os.path.join(data_directory, "*.txt"))

    if not txt_files:
        print(f"No .txt files found in {data_directory}/")
        return

    print(f"Analyzing data in: {data_directory}/")
    print(f"Found {len(txt_files)} batch files")
    print("=" * 50)

    total_characters = 0
    total_words = 0
    total_lines = 0
    file_sizes_mb = []

    # Process each file
    for i, file_path in enumerate(txt_files, 1):
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                text = f.read()

            # Calculate metrics for this file
            char_count = len(text)
            word_count = len(text.split())
            line_count = len(text.splitlines())
            size_mb = len(text.encode('utf-8')) / (1024 * 1024)

            # Add to totals
            total_characters += char_count
            total_words += word_count
            total_lines += line_count
            file_sizes_mb.append(size_mb)


        except Exception as e:
            print(f"Error reading {file_path}: {e}")

    # Print total statistics
    print("TOTAL DATA STATISTICS:")
    print(f"Total characters: {total_characters:,}")
    print(f"Total words: {total_words:,}")
    print(f"Total lines: {total_lines:,}")
    print(f"Total size: {sum(file_sizes_mb):.2f} MB")



    # Estimate tokens (rough approximation: 1 token ≈ 4 characters)
    estimated_tokens = total_characters / 4
    print(f"Estimated tokens (GPT): {estimated_tokens:,.0f}")

    return {
        'total_characters': total_characters,
        'total_words': total_words,
        'total_lines': total_lines,
        'total_size_mb': sum(file_sizes_mb),
        'estimated_tokens': estimated_tokens,
        'file_count': len(txt_files)
    }

# Run the analysis
if __name__ == "__main__":
    # Change this path if your data is in a different directory
    data_stats = calculate_total_data_length("gutenberg_science")

    # Simple version like the one demonstrated in class
    print("\n" + "="*30)
    print("SIMPLE LENGTH CHECK:")
    print("="*30)

    # Concatenate all files and show total length
    all_text = ""
    for file_path in glob.glob("gutenberg_science/*.txt"):
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                all_text += f.read()
        except:
            continue

    print("Length of data in letters or characters:")
    print(f"{len(all_text):,}")

Analyzing data in: gutenberg_science/
Found 297 batch files
TOTAL DATA STATISTICS:
Total characters: 1,426,945,009
Total words: 234,816,332
Total lines: 19,840,343
Total size: 1370.09 MB
Estimated tokens (GPT): 356,736,252

SIMPLE LENGTH CHECK:
Length of data in letters or characters:
1,426,945,009
